# Convert YOLO to Pascal VOC XML Format

This notebook converts YOLO format annotations to Pascal VOC XML format for compatibility with Pascal VOC tools and frameworks.

## Overview
The notebook:
1. Reads YOLO format label files (normalized coordinates from 0-1)
2. Loads corresponding images to determine image dimensions
3. Converts normalized YOLO coordinates to pixel-based coordinates
4. Generates Pascal VOC compatible XML annotations
5. Saves XML files for each image in the output directory

## Use Cases
- Convert YOLO annotations for use with Pascal VOC tools and frameworks
- Migrate annotation formats between different object detection pipelines
- Create XML annotations compatible with legacy systems
- Prepare datasets for multi-framework training workflows

## Requirements
- OpenCV (cv2)
- NumPy
- lxml
- CellProcessor module

## Documentation

### Coordinate System Conversion
**YOLO Format:**
- `class_id center_x center_y width height` (all normalized 0-1)
- Coordinates are relative to image dimensions

**Pascal VOC XML Format:**
- `xmin xmax ymin ymax` (all in pixel coordinates)
- Absolute pixel positions from top-left origin

### Pascal VOC XML Structure Reference
```xml
<annotation>
  <folder>Path</folder>
  <filename>image.png</filename>
  <source>
    <database>Cell Death Detection Database</database>
  </source>
  <size>
    <width>640</width>
    <height>480</height>
    <depth>3</depth>
  </size>
  <object>
    <name>0</name>
    <pose>Unspecified</pose>
    <truncated>0</truncated>
    <difficult>0</difficult>
    <bndbox>
      <xmin>100</xmin>
      <ymin>150</ymin>
      <xmax>200</xmax>
      <ymax>250</ymax>
    </bndbox>
  </object>
</annotation>
```

### Configuration Options
- **SPLIT**: Dataset split to process ("train", "val", or "test")
- **PATH_TO_DATASET**: Root directory containing images and labels subdirectories
- **YOLO_CLASSES**: List of class names (must match model classes)

### Output
- XML files are saved in `{PATH_TO_DATASET}/xml_outputs/` directory
- Filenames match the original image names with `.xml` extension
- Each XML file contains complete annotation metadata

## Import Required Libraries

In [61]:
# Import required libraries
import os
import cv2
from lxml.etree import Element, SubElement, tostring
import numpy as np
from CellProcessor import use_dataset, list_dataset

## Define Conversion Functions

Functions to convert coordinates and generate XML annotations.

In [62]:
def unconvert(class_id, width, height, x, y, w, h):
    """
    Convert YOLO normalized coordinates to Pascal VOC pixel coordinates.
    
    Coordinate System Conversion:
    - YOLO format: center_x, center_y, width, height (all normalized 0-1)
    - Pascal VOC format: xmin, xmax, ymin, ymax (all in pixels)
    
    Parameters:
        class_id (int): Class index
        width (int): Image width in pixels
        height (int): Image height in pixels
        x (float): Normalized center x coordinate (0-1)
        y (float): Normalized center y coordinate (0-1)
        w (float): Normalized width (0-1)
        h (float): Normalized height (0-1)
    
    Returns:
        tuple: (class_id, xmin, xmax, ymin, ymax) in pixels
    """
    xmax = int((x * width) + (w * width) / 2.0)
    xmin = int((x * width) - (w * width) / 2.0)
    ymax = int((y * height) + (h * height) / 2.0)
    ymin = int((y * height) - (h * height) / 2.0)
    class_id = int(class_id)
    return (class_id, xmin, xmax, ymin, ymax)

## Convert YOLO to Pascal VOC XML

Process all images and labels, generating Pascal VOC XML annotations.

In [63]:
def xml_transform(root, SPLIT, classes):
    """
    Transform YOLO annotations to Pascal VOC XML format.
    
    Processes all images and labels in the specified dataset split,
    converting YOLO format annotations to Pascal VOC XML format.
    
    Parameters:
        root (str): Root path to the dataset
        classes (list): List of class names indexed by class ID
    """
    # Define paths
    class_path = os.path.join(root, 'labels/' + SPLIT)
    annopath = os.path.join(root, 'labels/' + SPLIT, '%s.txt')
    imgpath = os.path.join(root, 'images/' + SPLIT, '%s.png')
    outpath = os.path.join(root, 'xml_outputs', '%s.xml')

    # Create output directory
    os.makedirs(os.path.join(root, 'xml_outputs'), exist_ok=True)
    
    # Get list of label files
    files = os.listdir(class_path)
    
    # Remove system files
    if '.DS_Store' in files:
        files.remove('.DS_Store')
    
    # Extract image IDs from filenames
    ids = [x.split('.')[0] for x in files]
    
    print(f"Starting conversion of {len(ids)} images...\n")
    
    processed = 0
    skipped = 0
    
    # Process each image
    for idx, img_id in enumerate(ids):
        # Skip special files
        if img_id == "classes":
            continue
        
        # Skip if XML already exists
        if os.path.exists(outpath % img_id):
            skipped += 1
            continue
        
        img_full_path = imgpath % img_id
        
        # Read image
        img = cv2.imread(img_full_path)
        if img is None:
            print(f"⚠ Warning: Could not read image {img_id}")
            continue
        
        height, width, channels = img.shape
        
        # Create XML root element
        node_root = Element('annotation')
        
        # Add folder metadata
        node_folder = SubElement(node_root, 'folder')
        node_folder.text = os.path.join(PATH_TO_DATASET, 'images/' + SPLIT)
        
        # Add filename
        img_name = img_id + '.png'
        node_filename = SubElement(node_root, 'filename')
        node_filename.text = img_name
        
        # Add source metadata
        node_source = SubElement(node_root, 'source')
        node_database = SubElement(node_source, 'database')
        node_database.text = 'Cell Death Detection Database'
        
        # Add image dimensions
        node_size = SubElement(node_root, 'size')
        node_width = SubElement(node_size, 'width')
        node_width.text = str(width)
        node_height = SubElement(node_size, 'height')
        node_height.text = str(height)
        node_depth = SubElement(node_size, 'depth')
        node_depth.text = str(channels)
        
        # Add segmented flag
        node_segmented = SubElement(node_root, 'segmented')
        node_segmented.text = '0'
        
        # Read and convert YOLO labels
        target = annopath % img_id
        if os.path.exists(target):
            label_norm = np.loadtxt(target).reshape(-1, 5)
            
            # Process each bounding box
            for label_idx in range(len(label_norm)):
                labels_conv = label_norm[label_idx]
                
                # Convert coordinates from YOLO to Pascal VOC format
                new_label = unconvert(
                    labels_conv[0], width, height,
                    labels_conv[1], labels_conv[2],
                    labels_conv[3], labels_conv[4]
                )
                
                # Create object element
                node_object = SubElement(node_root, 'object')
                
                # Add class name
                node_name = SubElement(node_object, 'name')
                node_name.text = classes[new_label[0]]
                
                # Add pose
                node_pose = SubElement(node_object, 'pose')
                node_pose.text = 'Unspecified'
                
                # Add truncated flag
                node_truncated = SubElement(node_object, 'truncated')
                node_truncated.text = '0'
                
                # Add difficult flag
                node_difficult = SubElement(node_object, 'difficult')
                node_difficult.text = '0'
                
                # Add bounding box coordinates
                node_bndbox = SubElement(node_object, 'bndbox')
                node_xmin = SubElement(node_bndbox, 'xmin')
                node_xmin.text = str(new_label[1])
                node_ymin = SubElement(node_bndbox, 'ymin')
                node_ymin.text = str(new_label[3])
                node_xmax = SubElement(node_bndbox, 'xmax')
                node_xmax.text = str(new_label[2])
                node_ymax = SubElement(node_bndbox, 'ymax')
                node_ymax.text = str(new_label[4])
            
            # Convert to string with pretty printing
            xml = tostring(node_root, pretty_print=True)
        else:
            print(f"⚠ Warning: No labels found for {img_id}")
            xml = tostring(node_root, pretty_print=True)
        
        # Write XML to file
        output_path = outpath % img_id
        with open(output_path, "wb") as f:
            f.write(xml)
        
        processed += 1
        if (processed + skipped) % 50 == 0:
            print(f"  Progress: {processed + skipped}/{len(ids)} images processed")
    
    output_dir = os.path.join(root, 'xml_outputs')
    print(f"\n✓ Conversion complete!")
    print(f"  Processed: {processed} images")
    print(f"  Skipped: {skipped} (already exist)")
    print(f"  Output directory: {output_dir}")

## Define Configuration

Set up class definitions and paths for the conversion process.

In [64]:
# List available datasets and select one to use
print("Available datasets:")
list_dataset()

Available datasets:


,ID,Cell_type,Death_type,Image_path,Description
0,1,MEF,Necroptosis,Data,Test


In [65]:
# Load dataset (using preset 1; modify as needed)
dataset = use_dataset(1)
print(f"Dataset: {dataset['Cell_type']} cells, {dataset['Death_type']} death type")
print(f"Image path: {dataset['Image_path']}")
print()

Dataset: MEF cells, Necroptosis death type
Image path: Data



In [66]:
# Configure paths and class definitions
PATH_TO_DATASET = "Data/final_Data_set"
SPLIT = "val"  # Options: "train", "val", "test"

# YOLO class list - first element is class 0 (background), second is death type
YOLO_CLASSES = ["1", dataset['Death_type']]

# Display configuration
print("Configuration:")
print(f"  Death type: {dataset['Death_type']}")
print(f"  Classes: {YOLO_CLASSES}")
print(f"  Dataset root: {PATH_TO_DATASET}")
print(f"  Split: {SPLIT}")

Configuration:
  Death type: Necroptosis
  Classes: ['1', 'Necroptosis']
  Dataset root: Data/final_Data_set
  Split: val


## Execute Conversion

In [68]:
# Run the conversion
xml_transform(PATH_TO_DATASET, SPLIT, YOLO_CLASSES)

Starting conversion of 304 images...

  Progress: 50/304 images processed
  Progress: 100/304 images processed
  Progress: 150/304 images processed
  Progress: 200/304 images processed
  Progress: 250/304 images processed
  Progress: 300/304 images processed

✓ Conversion complete!
  Processed: 304 images
  Skipped: 0 (already exist)
  Output directory: Data/final_Data_set/xml_outputs
